# MetricGAN-LwF Visualisierung

Dieses Notebook visualisiert Trainingsmetriken aus mehreren Runs.

Enthalten sind:
- Ein Metrik-Plot pro Run (alle gefundenen Metriken)
- Ein gemeinsamer PESQ-Vergleichsplot über alle Runs

In [ ]:
from pathlib import Path
import re
import warnings

import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
# Basisordner anpassen, falls deine Runs woanders liegen
BASE_RESULTS_DIR = Path("results/MetricGAN")

# Optional: explizite Run-Ordner angeben (überschreibt die Auto-Suche, wenn nicht leer)
MANUAL_RUN_DIRS = []

# Dateiname des SpeechBrain-Loggers
TRAIN_LOG_NAME = "train_log.txt"

In [ ]:
def discover_runs(base_dir: Path, manual_dirs=None, log_name="train_log.txt"):
    if manual_dirs is None:
        manual_dirs = []

    run_dirs = []

    for p in manual_dirs:
        rp = Path(p)
        if (rp / log_name).exists():
            run_dirs.append(rp)
        else:
            warnings.warn(f"Kein {log_name} in manuellem Run: {rp}")

    if not run_dirs and base_dir.exists():
        for cand in sorted(base_dir.iterdir()):
            if cand.is_dir() and (cand / log_name).exists():
                run_dirs.append(cand)

    return run_dirs


def _normalize_key(raw_key: str):
    key = raw_key.strip().lower()
    key = key.replace("_", " ").replace("-", " ")
    key = re.sub(r"\s+", " ", key)

    if key in {"epoch", "epoch loaded"}:
        return "epoch"
    if "pesq" in key:
        return "pesq"
    if "stoi" in key:
        return "stoi"
    if "si snr" in key or "sisnr" in key or "si-snr" in key:
        return "si_snr"
    if "csig" in key:
        return "csig"
    if "cbak" in key:
        return "cbak"
    if "covl" in key:
        return "covl"
    if "loss" in key:
        return "loss"

    return None


def parse_train_log(log_path: Path):
    text = log_path.read_text(encoding="utf-8", errors="ignore")

    # Key-Value Muster, z.B. 'Epoch: 3', 'pesq=2.45'
    kv_pattern = re.compile(r"([A-Za-z][A-Za-z0-9_\- ]{1,50})\s*[:=]\s*(-?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?)")

    rows = []
    block = {}

    def flush_block():
        nonlocal block
        if "epoch" in block and len(block) > 1:
            rows.append(block.copy())
        block = {}

    for raw_line in text.splitlines():
        line = raw_line.strip()

        if not line:
            flush_block()
            continue

        matches = kv_pattern.findall(line)
        if not matches:
            continue

        line_dict = {}
        for key, value in matches:
            norm = _normalize_key(key)
            if norm is None:
                continue
            line_dict[norm] = float(value)

        if not line_dict:
            continue

        if "epoch" in line_dict and "epoch" in block and len(block) > 1:
            flush_block()

        block.update(line_dict)

    flush_block()

    if rows:
        df = pd.DataFrame(rows).sort_values("epoch")
        df = df.drop_duplicates(subset=["epoch"], keep="last").reset_index(drop=True)
        return df

    # Fallback: TensorBoard Events lesen (falls train_log leer/anders formatiert)
    tb_dir = log_path.parent / "logs"
    if tb_dir.exists():
        try:
            from tensorboard.backend.event_processing import event_accumulator

            ea = event_accumulator.EventAccumulator(str(tb_dir))
            ea.Reload()
            scalar_tags = ea.Tags().get("scalars", [])

            wanted = ["pesq", "stoi", "si", "loss"]
            selected_tags = [t for t in scalar_tags if any(w in t.lower() for w in wanted)]
            if not selected_tags:
                return pd.DataFrame()

            rows_tb = {}
            for tag in selected_tags:
                norm = _normalize_key(tag.split("/")[-1])
                if norm is None:
                    continue

                for ev in ea.Scalars(tag):
                    step = float(ev.step)
                    rows_tb.setdefault(step, {"epoch": step})
                    rows_tb[step][norm] = float(ev.value)

            if rows_tb:
                return pd.DataFrame(list(rows_tb.values())).sort_values("epoch").reset_index(drop=True)

        except Exception as exc:
            warnings.warn(f"TensorBoard-Fallback fehlgeschlagen ({log_path}): {exc}")

    return pd.DataFrame()

In [ ]:
run_dirs = discover_runs(BASE_RESULTS_DIR, MANUAL_RUN_DIRS, TRAIN_LOG_NAME)

if not run_dirs:
    print("Keine Runs gefunden. Setze BASE_RESULTS_DIR oder MANUAL_RUN_DIRS.")
else:
    print("Gefundene Runs:")
    for r in run_dirs:
        print(f"- {r}")

In [ ]:
run_data = {}

for run_dir in run_dirs:
    log_path = run_dir / TRAIN_LOG_NAME
    df = parse_train_log(log_path)

    if df.empty:
        warnings.warn(f"Keine Metriken gefunden in {log_path}")
        continue

    run_data[run_dir.name] = df

print(f"Eingelesene Runs mit Metriken: {len(run_data)}")
for run_name, df in run_data.items():
    print(f"- {run_name}: Spalten={list(df.columns)}, Epochen={len(df)}")

In [ ]:
# 1) Pro Run: Metrikentwicklung
for run_name, df in run_data.items():
    metric_cols = [c for c in df.columns if c != "epoch"]
    if not metric_cols:
        continue

    fig, ax = plt.subplots(figsize=(11, 5))
    for col in metric_cols:
        ax.plot(df["epoch"], df[col], marker="o", linewidth=1.8, markersize=3, label=col)

    ax.set_title(f"Run: {run_name} | Metrikentwicklung", fontsize=13)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Wert")
    ax.legend(loc="best", ncols=min(4, max(1, len(metric_cols))))
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

In [ ]:
# 2) Gemeinsamer Plot: alle PESQ-Kurven
fig, ax = plt.subplots(figsize=(11, 5))
found_pesq = False

for run_name, df in run_data.items():
    if "pesq" not in df.columns:
        continue
    found_pesq = True
    ax.plot(df["epoch"], df["pesq"], marker="o", linewidth=2.0, markersize=3, label=run_name)

if found_pesq:
    ax.set_title("PESQ-Vergleich über alle Läufe", fontsize=13)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("PESQ")
    ax.legend(loc="best")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    plt.close(fig)
    print("Keine PESQ-Spalte in den eingelesenen Runs gefunden.")

In [ ]:
# Optional: zusammengeführte Tabelle exportieren
export_rows = []
for run_name, df in run_data.items():
    temp = df.copy()
    temp.insert(0, "run", run_name)
    export_rows.append(temp)

if export_rows:
    merged = pd.concat(export_rows, ignore_index=True)
    out_csv = Path("run_metrics_merged.csv")
    merged.to_csv(out_csv, index=False)
    print(f"Exportiert: {out_csv.resolve()}")
else:
    print("Nichts zu exportieren.")